# <font color="#003660">Session 9: How to Evaluate LLMs?</font>

# <font color="#003660">Evaluate using Metrics</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>

<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will know how to evaluate LLMs using Metrics. <br>
        ... will know how to apply exact match to a JSON-Problem.
    </font>
</div>
</p>

The following content is heavily inspired by the following excellent sources:

* [LangChain Academy](https://academy.langchain.com/)
* [LangChain Docs (Python)](https://python.langchain.com/)
* [LangSmith Evaluation](https://docs.smith.langchain.com/evaluation)
* [Chang et al. (2024)](https://doi.org/10.1145/3641289)

This Notebook is based on LangSmith Evaluation, but we refrain from using LangSmith, because it is an API driven online-tool.

# Setting up LangChain-Ollama

First we Setup Langchain-Ollama, which will help us to generate the structured outputs.

In [ ]:
!pip install -U langchain langchain-community langchain-openai langchain-ollama

Today we will setup our own ollama server. We can do this directly in Google Colab.

First we need to install the pciutil package (to let ollama automatically detect GPU) and ollama. Just run the code below.

In [ ]:
# with this linux package, ollama can then detect GPU, if available
!sudo apt-get install -y pciutils

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

The next chunk is to start ollama locally as a subprocess in the background. (Even if ollama tells you that it has started the server, it has not.)

In [ ]:
import subprocess

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)


Now we will have to download both models for this session. Run the code below.

In [ ]:
!ollama pull jina/jina-embeddings-v2-base-de # This should run fast
!ollama pull gpt-oss # this may take a 1-5 minutes depending on the interent connection of the colab environment

In [ ]:
import os
import re
import ast
import time
import json
import signal
import requests

import numpy as np

from langchain_openai import ChatOpenAI

from langchain_ollama import OllamaEmbeddings
from sklearn.metrics.pairwise import cosine_similarity


from typing import List, Optional, Union
from pydantic import BaseModel, Field

from tqdm.notebook import tqdm

# An Easy Exact Match Example

First we start with exact match on mail extraction.

Exact match of the values ``a`` and ``b``is calculated by ``1 if a == b else 0`` ([Chang et al., 2024](https://doi.org/10.1145/3641289))

At first we need to setup the llm with structured output.

## Setup the Model

In [ ]:
LLM_CONFIG = {
    "base_url": "http://localhost:11434/v1",
    "model": "gpt-oss",
    "temperature": 0.0,
    "api_key": "ollama",
    "seed": 42,
}

llm = ChatOpenAI(
    **LLM_CONFIG
)

## Setup Structured Output Class and Model

In [ ]:
class EmailAddressModel(BaseModel):
    email_address: str = Field(
        description="The email address."
    )

In [ ]:
llm_with_structured_output = llm.with_structured_output(
    EmailAddressModel,
    method="json_schema"
)

## Test Structured Output

In [ ]:
query = "My Name is Oliver Müller you can contact me using oliver.mueller@uni-paderborn.de"
response = llm.invoke(
    f"""Extract the email address from this text: {
        query
    }""")
print(response.model_dump_json(indent=4))

## Calculate Exact Match

In [ ]:
def exact_match(address_infos_test):
    for address_info in address_infos_test:
        resp = llm_with_structured_output.invoke(f"Extract the email address from this text: {address_info}").model_dump()
        if address_info[1] == resp.get("email_address", ""):
            print(f"✅ {resp}")
        else:
            print(f"❌ {resp}")

In [ ]:
address_infos_test = [
    ["My Name is Oliver Müller you can contact me using oliver.mueller@uni-paderborn.de", "oliver.mueller@uni-paderborn.de"],
    ["Hey, here is Sascha, please contact me under sascha.kaltenpoth@uni-paderborn.de", "sascha.kaltenpoth@uni-paderborn.de",],
    ["Hello it's Dirk, you should have received an email from dirk.leffrang@uni-paderborn.de. Please answer the email or, if you haven't received the mail write me.", "dirk.leffrang@uni-paderborn.de"],
    ["Hello its Max Mustermann, you can contact me under max.mustermann@muster-firm.com if you want to here about our latest products.", "max.mustermann@muster-firm.com"],
]

exact_match(address_infos_test)

Wow that was easy. But what happens, when we have some stuff like the fuzzy problem descriptions below.

# A resource-efficient Way to Fuzzy matching - Check Similarities

## Setup the Model

In [ ]:
LLM_CONFIG = {
    "base_url": "http://localhost:11434/v1",
    "model": "gpt-oss",
    "temperature": 0.0,
    "api_key": "ollama",
    "seed": 42,
}

llm = ChatOpenAI(
    **LLM_CONFIG
)

## Setup Structured Output Class and Model

In [ ]:
class EmailAddressModel(BaseModel):
    email_address: str = Field(
        description="The email address."
    )
    problem_description: str = Field(
        description="The problem description."
    )

In [ ]:
llm_with_structured_output = llm.with_structured_output(
    EmailAddressModel,
    method="json_schema"
)

## Test Structured Output

In [ ]:
query = "My Name is Oliver Müller. My Bank account was hacked. You can contact me using oliver.mueller@uni-paderborn.de"
response = llm_with_structured_output.invoke(
    f"""Extract the email address and problem description from this text: {
        query
    }""")
print(response.model_dump_json(indent=4))

## Calculate Similarity

We start with a small helper function that calculates the cosine similarity between two texts. Why do we do that? Simple answer: this is the most simple way to check if there is a fuzzy match [(Ganesan et al., 2024)](https://doi.org/10.48550/arXiv.2401.08688)

In [ ]:
# small helper function for fuzzy match
def compare_cosine_similarity(text1: str, text2: str) -> float:
    """
    Calculates the cosine similarity between two texts using Ollama's jina-embeddings-v2-base-de model.

    Args:
        text1: The first text string.
        text2: The second text string.

    Returns:
        The cosine similarity score between the two texts.
    """
    # Initialize the embedding model pointing to the local Ollama server
    embeddings_model = OllamaEmbeddings(
        base_url="http://localhost:11434",
        model="jina/jina-embeddings-v2-base-de",
    )

    # Generate embeddings for the texts
    embedding1 = embeddings_model.embed_documents([text1])
    embedding2 = embeddings_model.embed_documents([text2])

    # Calculate cosine similarity
    # cosine_similarity expects 2D arrays, so reshape the embeddings
    similarity = cosine_similarity(np.array(embedding1).reshape(1, -1), np.array(embedding2).reshape(1, -1))[0][0]

    return similarity

## Your ToDo: Implement Similarity Calculation

In [ ]:
def similarity_match(address_infos_test, threshold=0.5):
    for address_info in address_infos_test:
        resp = llm_with_structured_output.invoke(f"Extract the email address and problem description from this text: {address_info}").model_dump()
        similarity = compare_cosine_similarity(address_info[2], resp.get("problem_description", ""))
        if similarity > threshold:
            print(f"✅ ({round(similarity, 4)}) {resp}")
        else:
            print(f"❌ ({round(similarity, 4)}) {resp}")

In [ ]:
address_infos_test = [
    [
        "My Name is Oliver Müller. My Bank account was hacked. You can contact me using oliver.mueller@uni-paderborn.de",
        "oliver.mueller@uni-paderborn.de",
        "My Bank account was hacked."
    ],
    [
        "Hey, here is Sascha. I have a question regarding my university enrollment. Please contact me under sascha.kaltenpoth@uni-paderborn.de",
        "sascha.kaltenpoth@uni-paderborn.de",
        "I have a question regarding my university enrollment."
    ],
    [
        "Hello it's Dirk, I sent an email earlier and haven't heard back yet. It should have come from dirk.leffrang@uni-paderborn.de, just in case you need it again.",
        "dirk.leffrang@uni-paderborn.de",
        "Dirk is waiting for a response to my previous email."
    ],
    [
        "Hi, this is Max Mustermann. I'm writing because of our recent talk about new products. If needed, you can reach me at max.mustermann@muster-firm.com.",
        "max.mustermann@muster-firm.com",
        "Max wants to present information about our latest products."
    ],
]

similarity_match(address_infos_test)

# Real Fuzzy Matching with LLMs

We can use LLMs as a judge to assess the correctness of the extracted fuzzy outputs, but as we will see this can also lead to wrong decisions ([Verga et al., 2024](https://arxiv.org/pdf/2404.18796)).

## Setup the Model

In [ ]:
LLM_CONFIG = {
    "base_url": "http://localhost:11434/v1",
    "model": "gpt-oss",
    "temperature": 0.0,
    "api_key": "ollama",
    "seed": 42,
}

llm = ChatOpenAI(
    **LLM_CONFIG
)

## Setup Structured Output Class and Model

In [ ]:
class EmailAddressModel(BaseModel):
    email_address: str = Field(
        description="The email address."
    )
    problem_description: str = Field(
        description="The problem description."
    )

In [ ]:
llm_with_structured_output = llm.with_structured_output(
    EmailAddressModel,
    method="json_schema"
)

## Test Structured Output

In [ ]:
query = "My Name is Oliver Müller. My Bank account was hacked. You can contact me using oliver.mueller@uni-paderborn.de"
response = llm_with_structured_output.invoke(
    f"""Extract the email address and a short description of the problem from this text: {
        query
    }""")
print(response.model_dump_json(indent=4))

## Define Fuzzy Matching Functions

In [ ]:
def compare_with_llm(text1: str, text2: str) -> bool:
    resp = llm.invoke(
        f"""Compare the two texts below. If they have the same meaning return TRUE, if the differ strongy, then return FALSE. Return nothing else, only 'TRUE' or 'FALSE'.
        Text 1: {text1}
        Text 2: {text2}
        Now compare the two texts:
        """
    ).content
    if resp == "TRUE":
        return True
    else:
        return False

Your ToDo:

In [ ]:
def fuzzy_match_with_llm(address_infos_test, threshold=0.5):
    for address_info in address_infos_test:
        resp = llm_with_structured_output.invoke(f"Extract the email address and problem description from this text: {address_info[0]}").model_dump()
        resp_validation = compare_with_llm(address_info[2], resp.get("problem_description", ""))
        if resp_validation:
            print(f"✅ {resp}")
        else:
            print(f"❌ {resp}")

In [ ]:
address_infos_test = [
    [
        "My Name is Oliver Müller. My Bank account was hacked. You can contact me using oliver.mueller@uni-paderborn.de",
        "oliver.mueller@uni-paderborn.de",
        "My Bank account was hacked."
    ],
    [
        "Hey, here is Sascha. I have a question regarding my university enrollment. Please contact me under sascha.kaltenpoth@uni-paderborn.de",
        "sascha.kaltenpoth@uni-paderborn.de",
        "I have a question regarding my university enrollment."
    ],
    [
        "Hello it's Dirk, I sent an email earlier and haven't heard back yet. It should have come from dirk.leffrang@uni-paderborn.de, just in case you need it again.",
        "dirk.leffrang@uni-paderborn.de",
        "Dirk is waiting for a response to his previous email."
    ],
    [
        "Hi, this is Max Mustermann. I'm writing because of our recent talk about my company's new products. If product information needed, you can reach me at max.mustermann@muster-firm.com.",
        "max.mustermann@muster-firm.com",
        "Max can provide information about his company's new products if needed."
    ],
    [
        "Hi, this is Max Mustermann. I'm writing because of our recent talk about new products. If needed, you can reach me at max.mustermann@muster-firm.com.",
        "max.mustermann@muster-firm.com",
        "Max wants to kill the ceo."
    ],
    [
        "Hi, this is Max Mustermann. I'm writing because of our recent talk about new products. If needed, you can reach me at max.mustermann@muster-firm.com.",
        "max.mustermann@muster-firm.com",
        "Max wants to get some information about our latest products."
    ],
]

fuzzy_match_with_llm(address_infos_test)